[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/motor_loop/motor_loop.ipynb)

# Loop Subdivision of Motors

A Loop step refines a triangle mesh by a sparse map on its vertex sites: every vertex of the finer mesh is a fixed average of vertices of the coarser one, and the shares depend only on how the mesh is connected. The map does not care what sits at the sites. Here every vertex carries a motor, a rigid placement of a frame, and each step applies the same map to the motors and then normalizes every averaged motor back to a motor: average in the algebra, return to the group. A ribbon wound up by a twist shows what that buys. Refined as points it wrings itself thin, because the average of two positions cuts across the turn between them; refined as motors it keeps its width, because the normalized average of two frames is the frame turned halfway, out on the arc between them.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from numga.sparse import SparseExtensor
from examples import instantiate
from examples.animation import show_animation
from examples.surfaces.motor_loop import render

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()
Line = ga.gatype.bivector()
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
Mesh = instantiate("examples.mesh", ga).Mesh


# The ribbon's control mesh: its vertices along and across, its length and width, and its four corners,
# pinned.
along_count, across_count = 5, 3
length, width = 4.0, 1.2
corners = [0, across_count - 1, along_count * across_count - across_count, along_count * across_count - 1]
# The turn of the far end at the top of the wind-up, the near end held; how far each frame turns either way
# about its own axis as the ribbon lies flat, and the seed of those axes and of the turns' phases.
wound_turn = 3 * np.pi
turning_angle = np.pi / 3
turning_seed = 2
# The frames of each animation and the duration of each, the Loop steps taken, and the length of the drawn
# frame axes.
frames_count = 48
duration_ms = 60
depth = 3
axis_length = 0.3

# How far along and across the ribbon each control vertex lies, row by row, and its faces: two triangles
# per cell of the grid, all turning the same way.
along, across = (grid.reshape(-1) for grid in np.meshgrid(np.linspace(-length / 2, length / 2, along_count),
                                                           np.linspace(-width / 2, width / 2, across_count), indexing="ij"))
index = np.arange(along_count * across_count).reshape(along_count, across_count)
cell, right, up, opposite = index[:-1, :-1], index[:-1, 1:], index[1:, :-1], index[1:, 1:]
control_faces = np.concatenate([np.stack([cell, right, opposite], axis=-1).reshape(-1, 3),
                                np.stack([cell, opposite, up], axis=-1).reshape(-1, 3)])
pinned = np.isin(np.arange(along_count * across_count), corners)

## 1. One map, any contents

`Mesh.loop` builds the map of one Loop step from the faces alone, with the pinned corners keeping themselves: a sparse map from the vertex sites of the coarser mesh to those of the finer one, its cells scalars. `refinements` steps the mesh of motors, normalizing every averaged motor after each step, since a weighted sum of motors is no motor. The points the motors carry go through the same maps, and with nothing to do between the steps, the maps compose into one. The two surfaces, side by side, are the whole comparison. A frame is drawn as its origin and the ends of its three axes, and the surface of the motors is the origin each one carries, `motor >> origin`.

In [ ]:
def turn(plane: Line, angle: np.ndarray) -> Motor:
    """The turn by an angle in a plane through the origin."""
    return (plane * (-angle / 2)).exp()


def slide(direction: Plane, distance: np.ndarray) -> Motor:
    """The translation by a distance along the normal of a plane through the origin."""
    return (direction * mv.w * (distance / 2)).exp()


def ribbon(along: np.ndarray, across: np.ndarray, angle: np.ndarray) -> Motor:
    """Frames on a ribbon down the z axis: slid along it and out across it along x, then turned about the
    axis by an angle."""
    return turn(mv.xy, angle) * slide(mv.z, along) * slide(mv.x, across)


def refinements(mesh: Mesh, pinned: np.ndarray, depth: int) -> Iterator[tuple[SparseExtensor, Mesh]]:
    """Each of so many Loop steps on a mesh of motors, with the vertices pinned staying where they are: the
    averaging map, built once from the faces, and the finer mesh it gives, every averaged motor normalized
    back to a motor."""
    for _ in range(depth):
        step = mesh.loop(pinned)                                               # [finer, coarser] Scalar
        mesh, pinned = Mesh((step * mesh.vertices).normalized(), mesh.split_faces), mesh.split_pins(pinned)
        yield step, mesh


markers = stack([mv.w.dual(), mv.point([axis_length, 0.0, 0.0]), mv.point([0.0, axis_length, 0.0]),
                 mv.point([0.0, 0.0, axis_length])])                           # [axes + 1] Point
origin = markers[0]                                                            # [] Point


def refined(motors: Motor) -> tuple[Point, Point, np.ndarray]:
    """The control frames of motors at the vertices of the ribbon, and the surface refined two ways: as the
    points the motors carry, through the steps composed into one map, and as the motors themselves,
    normalized after every step; with the faces of the surfaces."""
    steps, meshes = zip(*refinements(Mesh(motors, control_faces), pinned, depth))
    points = reduce(lambda total, step: step * total, steps) * (motors >> origin)   # [frames] Point[refined]
    surfaces = stack([points, meshes[-1].vertices >> origin], axis=1)          # [frames, schemes] Point[refined]
    return motors[:, None] >> markers, surfaces, meshes[-1].faces              # [frames, axes + 1] Point[vertices]

## 2. Winding up the ribbon

One end of the ribbon is held and the other wound one and a half turns and back, the twist building towards the wound end as the square of the distance. The points, on the left, are averaged as positions: two frames far apart in turn have their average close to the axis, and the ribbon wrings itself into a strand. The motors, on the right, are averaged as placements, and every average lands turned partway between its neighbours, at their distance from the axis. The bold markers are the control frames, x red, y green and z blue, over the dashed control net.

In dual-quaternion notation a motor is a unit dual quaternion and each step is dual-quaternion linear blending followed by normalization; refining the points is linear blend skinning, and the strand is its candy-wrapper artifact.

In [ ]:
wound = wound_turn * (1 - np.cos(np.linspace(0, 2 * np.pi, frames_count, endpoint=False))) / 2   # [frames]
held = ((along + length / 2) / length) ** 2                                    # [vertices]
winding = ribbon(along, across, wound[:, None] * held).field()                 # [frames] Motor[vertices]
control, surfaces, faces = refined(winding)                                    # [frames, schemes] Point[refined]

show_animation(render.animate(control, surfaces, control_faces, faces), "motor_loop_ribbon", duration_ms)

## 3. Turning the frames in place

Hold the ribbon flat and turn every frame back and forth about an axis of its own through its origin, each with its own phase. No control point moves, so neither does the surface of the points. The surface of the motors does: the average of two frames moves along the screw motion between them, and where neighbouring frames turn against each other that motion bends, bowing the surface out of the chord. A turn shared by all frames changes nothing, since it carries every origin to itself; only the turns between neighbours bend the surface, so the frames are no tangents the surface must follow, but a bending between neighbours.

In [ ]:
rng = np.random.default_rng(turning_seed)
axes = mv(Line.derive.nondegenerate, rng.normal(size=(len(along), 3))).normalized()   # [vertices] Line
phases = rng.uniform(0, 2 * np.pi, len(along))
angles = turning_angle * np.sin(np.linspace(0, 2 * np.pi, frames_count, endpoint=False)[:, None] + phases)   # [frames, vertices]
turning = (ribbon(along, across, 0.0) * turn(axes, angles)).field()           # [frames] Motor[vertices]
turning_control, turning_surfaces, turning_faces = refined(turning)            # [frames, schemes] Point[refined]

show_animation(render.animate(turning_control, turning_surfaces, control_faces, turning_faces), "motor_loop_turning", duration_ms)

In [ ]:
# checks
(points_rest, motors_rest), (points_peak, motors_peak) = render.reach(surfaces[[0, frames_count // 2]])
# Untwisted the two agree; wound up, the points wring the ribbon towards its axis and the motors do not.
assert abs(points_rest - motors_rest) < 1e-12
assert points_peak < 0.7 * points_rest and motors_peak >= motors_rest
# Turning the frames in place leaves the points alone and bends the motors.
turning_points, turning_motors = turning_surfaces[:, 0], turning_surfaces[:, 1]   # [frames] Point[refined]
assert render.separation(turning_points, turning_points[:1]) < 1e-12
assert render.separation(turning_motors, turning_points) > 0.05
# A turn common to every frame changes nothing.
flat = ribbon(along, across, 0.0)                                              # [vertices] Motor
common = refined(stack([flat, flat * turn(mv.yz, 0.7)]).field())[1][:, 1]     # [cases] Point[refined]
assert render.separation(common[0], common[1]) < 1e-12